In [11]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
import random
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import ConcatDataset
!pip install optuna
import optuna
import sys
sys.path.append('/content/')

from nets import *
from data import *
from train import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.3 MB/s eta 0:00:00


In [12]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False
# torch.use_deterministic_algorithms(True)

set_seed(SEED)

# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # wymagane przez niektóre operacje CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN = False
DATA_FOLDER = "train/"

print(device)

cuda


In [5]:
# from google.colab import files
# files.upload()

Saving train.zip to train.zip


In [ ]:
# !unzip train.zip

## Augmentacja danych

### Blur

In [13]:
transform_blur = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.GaussianBlur(kernel_size=5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset_blur = load_data(data_folder=DATA_FOLDER, transform=transform_blur)
merged_blur_dataset = ConcatDataset([dataset_blur, dataset_blur])
train_dataset_blur, val_dataset_blur = split_dataset(merged_blur_dataset, split_ratio=0.8)
train_loader_blur, val_loader_blur = define_dataloaders(train_dataset_blur, val_dataset_blur, batch_size=32)

net_blur = Net().to(device)
criterion_blur = nn.CrossEntropyLoss()
optimizer_blur = optim.Adam(net_blur.parameters(), lr=0.001)

# TRAIN = True
if TRAIN:
    loss_hist_blur, train_eval_hist_blur, val_eval_hist_blur = train_model(net_blur, train_loader_blur, val_loader_blur, criterion_blur, optimizer_blur, get_accuracy, device, 5)
    plot_training_chart(loss_hist_blur, train_eval_hist_blur, val_eval_hist_blur)

### Cropowanie

In [ ]:
transform_crop = transforms.Compose([
    transforms.RandomResizedCrop(size=(64, 64), scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset_crop = load_data(data_folder=DATA_FOLDER, transform=transform_crop)
merged_crop_dataset = ConcatDataset([dataset_crop, dataset_crop])
train_dataset_crop, val_dataset_crop = split_dataset(merged_crop_dataset, split_ratio=0.8)
train_loader_crop, val_loader_crop = define_dataloaders(train_dataset_crop, val_dataset_crop, batch_size=32)

net_crop = Net().to(device)
criterion_crop = nn.CrossEntropyLoss()
optimizer_crop = optim.Adam(net_crop.parameters(), lr=0.001)

if TRAIN:
    loss_hist_crop, train_eval_hist_crop, val_eval_hist_crop = train_model(net_crop, train_loader_crop, val_loader_crop, criterion_crop, optimizer_crop, get_accuracy, device, 5)
    plot_training_chart(loss_hist_crop, train_eval_hist_crop, val_eval_hist_crop)

### Zmiana jasności/saturacji/contrastu...

In [ ]:
transform_color_jitter = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset_color_jitter = load_data(data_folder=DATA_FOLDER, transform=transform_color_jitter)
merged_color_jitter_dataset = ConcatDataset([dataset_color_jitter, dataset_color_jitter])
train_dataset_color_jitter, val_dataset_color_jitter = split_dataset(merged_color_jitter_dataset, split_ratio=0.8)
train_loader_color_jitter, val_loader_color_jitter = define_dataloaders(train_dataset_color_jitter, val_dataset_color_jitter, batch_size=32)

net_color_jitter = Net().to(device)
criterion_color_jitter = nn.CrossEntropyLoss()
optimizer_color_jitter = optim.Adam(net_color_jitter.parameters(), lr=0.001)

if TRAIN:
    loss_hist_color_jitter, train_eval_hist_color_jitter, val_eval_hist_color_jitter = train_model(net_color_jitter, train_loader_color_jitter, val_loader_color_jitter, criterion_color_jitter, optimizer_color_jitter, get_accuracy, device, 5)
    plot_training_chart(loss_hist_color_jitter, train_eval_hist_color_jitter, val_eval_hist_color_jitter)

### Obracanie

In [ ]:
transform_rotate = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomRotation(degrees=30),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset_rotate = load_data(data_folder=DATA_FOLDER, transform=transform_rotate)
merged_rotate_dataset = ConcatDataset([dataset_rotate, dataset_rotate])
train_dataset_rotate, val_dataset_rotate = split_dataset(merged_rotate_dataset, split_ratio=0.8)
train_loader_rotate, val_loader_rotate = define_dataloaders(train_dataset_rotate, val_dataset_rotate, batch_size=32)

net_rotate = Net().to(device)
criterion_rotate = nn.CrossEntropyLoss()
optimizer_rotate = optim.Adam(net_rotate.parameters(), lr=0.001)

if TRAIN:
    loss_hist_rotate, train_eval_hist_rotate, val_eval_hist_rotate = train_model(net_rotate, train_loader_rotate, val_loader_rotate, criterion_rotate, optimizer_rotate, get_accuracy, device, 5)
    plot_training_chart(loss_hist_rotate, train_eval_hist_rotate, val_eval_hist_rotate)

### Wymazywanie

In [ ]:
transform_erase = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3)),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset_erase = load_data(data_folder=DATA_FOLDER, transform=transform_erase)
merged_erase_dataset = ConcatDataset([dataset_erase, dataset_erase])
train_dataset_erase, val_dataset_erase = split_dataset(merged_erase_dataset, split_ratio=0.8)
train_loader_erase, val_loader_erase = define_dataloaders(train_dataset_erase, val_dataset_erase, batch_size=32)

net_erase = Net().to(device)
criterion_erase = nn.CrossEntropyLoss()
optimizer_erase = optim.Adam(net_erase.parameters(), lr=0.001)

if TRAIN:
    loss_hist_erase, train_eval_hist_erase, val_eval_hist_erase = train_model(net_erase, train_loader_erase, val_loader_erase, criterion_erase, optimizer_erase, get_accuracy, device, 5)
    plot_training_chart(loss_hist_erase, train_eval_hist_erase, val_eval_hist_erase)

# Dostrajanie hiperparametrów

In [ ]:
# Łączenie wszystkich transformacji w jeden zbiór danych
clean_dataset = load_data(data_folder=DATA_FOLDER)
dataset_all_transforms = ConcatDataset([dataset_blur, dataset_crop, dataset_color_jitter, dataset_rotate, dataset_erase])

# Wartości do sprawdzenia
kernel_sizes = [[3, 5, 7], [3, 3, 3], [5, 5, 5], [7, 7, 7]]
n_channels = [16, 32, 64]
hidden_sizes = [32, 64, 128]

In [ ]:
def objective(trial):
    ks = trial.suggest_categorical("kernel_sizes", kernel_sizes)
    nc = trial.suggest_categorical("n_channels", n_channels)
    hs = trial.suggest_categorical("hidden_sizes", hidden_sizes)

    print(f"Testing with kernel_sizes={ks}, n_channels={nc}, hidden_size={hs}")
    net = Net(kernel_sizes=ks, n_channels=nc, hidden_size=hs).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=0.001)

    train_loader = DataLoader(dataset_all_transforms, batch_size=32, shuffle=True, num_workers=4)
    val, test = split_dataset(clean_dataset, split_ratio=0.5)
    val_loader, test_loader = define_dataloaders(val, test, batch_size=32)

    train_model(net, train_loader, val_loader, criterion, optimizer, get_accuracy, device, 5)

    return get_accuracy(net, test_loader, device)


In [ ]:
opt = optuna.create_study(direction="maximize")
opt.optimize(objective, n_trials=10)

print(opt.best_params)
print(opt.best_value)